# Phase 9 — LangGraph Agentic RAG

This notebook runs one question through the Phase 9 conditional `StateGraph`: intent analysis, guarded query handling, metadata construction, dense retrieval, the selected no-op reranking path, evidence grading, at most one retry, generation, and grounding verification.

It is intentionally a single-workflow trace, not the Phase 10 cross-version evaluation. Running it rebuilds only the guarded `phase9-*` namespace.

## Setup

Use the repository `.venv` kernel. The runner loads `.env` and connects to Ollama and Pinecone.

In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path

import yaml
from IPython.display import JSON, display

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in (Path.cwd(), Path.cwd().parent)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")


def run_command(command: list[object]) -> None:
    normalized = [str(part) for part in command]
    print(shlex.join(normalized))
    subprocess.run(normalized, cwd=PROJECT_ROOT, check=True)


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def read_yaml(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}


print(f"Project root: {PROJECT_ROOT}")
print(f"Kernel Python: {sys.executable}")


## Parameters

Change `QUESTION` as needed. External execution remains opt-in.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config" / "agentic_rag.yaml"
QUESTION = "What is the neighborhood potluck RSVP status?"
OUTPUT_PATH = PROJECT_ROOT / "evaluation" / "results" / "phase9_agentic" / "run.json"
RUN_EXPERIMENT = False  # Set to True, then run the execution cell.

print("Graph configuration:")
display(JSON(read_yaml(CONFIG_PATH)))

## Run one graph invocation

In [ ]:
COMMAND = [
    sys.executable,
    PROJECT_ROOT / "evaluation" / "run_agentic_rag.py",
    "--config", CONFIG_PATH,
    "--question", QUESTION,
    "--output", OUTPUT_PATH,
]

if RUN_EXPERIMENT:
    run_command(COMMAND)
else:
    print("Dry run. Set RUN_EXPERIMENT = True to execute:")
    print(shlex.join(str(part) for part in COMMAND))

## Inspect the final state and route

In [ ]:
if OUTPUT_PATH.exists():
    payload = read_json(OUTPUT_PATH)
    state = payload["state"]
    display(JSON({
        "question": payload["question"],
        "answer": state.get("answer"),
        "citations": state.get("citations"),
        "retrieval_attempts": state.get("retrieval_attempts"),
        "evidence_sufficient": state.get("evidence_sufficient"),
        "grounded": state.get("grounded"),
        "refusal_reason": state.get("refusal_reason"),
    }))
    print("Node trace:")
    display(JSON(state.get("node_trace", [])))
else:
    print(f"No graph trace yet: {OUTPUT_PATH}")

## Inspect retry and grounding details

In [ ]:
if OUTPUT_PATH.exists():
    state = read_json(OUTPUT_PATH)["state"]
    display(JSON({
        "query_history": state.get("query_history", []),
        "retrieval_history": state.get("retrieval_history", []),
        "evidence_history": state.get("evidence_history", []),
        "draft_answer": state.get("draft_answer"),
        "grounding_result": state.get("grounding_result"),
        "final_chunks": state.get("final_chunks", []),
    }))